### Examen 11/05/2026: Evaluación 3 Examen 2: Redes neuronales

<div style="border-style:groove;border-width:thin;padding:10px">
    <h4>NORMAS:</h4>
    <ul>
        <li>No se dispondrá de Internet durante el examen. Solo se podrán usar las presentaciones descargadas y una hoja escrita a mano por las dos caras.</li>
        <li>Queda prohibida cualquier comunicación entre vosotros o con el exterior.</li>
        <li>Deberá grabarse toda la sesión desde antes de recibir el examen hasta después de entregarlo utilizando OBS de forma local. La grabación se entregará a la vez que el examen. Quedan prohibidos los atajos de teclado del OBS</li> 
        <li>No se puede utilizar ninguna herramienta que asista a la programación. Ni basada en IA ni de otro tipo. Tampoco se podrá usar material hecho con IA, solo el material aportado por el profesor y el material hecho por el propio alumno (Sin IA).</li>
        <li>No se permite el uso de funciones genéricas que atajen distintos tipos de problemas. Solo se valorarán las soluciones concretas a lo que se pregunta. </li>
        <li>La entrega del examen se hará supervisada por el profesor y se anotará la hora exacta de entrega.</li>
        <li>Los móviles permanecerán durante toda la sesión sobre los PCs colocados con la pantalla hacia abajo.</li>
    </ul>
</div> 

<div style="border-style:groove;border-width:thin;padding:10px">
    <h4>EJERCICIO 1:</h4>
    <p>Importa el dataset 'ai_workforce_displacement_global_2020_2026.csv'. Contiene datos sobre despidos por IA en los últimos 6 años por países y por sectores. Debes entrenar una IA para predecir la columna objetivo: net_workforce_change_pct. Trata de conseguir los siguientes objetivos:</p>
    <ol>
        <li>Adecúa los datos para poder solucionar el problema con IA.</li>
        <li>Resuelve el problema con una red neuronal usando todas las técnicas de deep learning que tengan sentido y siguiendo un proceso correcto para solucionar el problema. Debéis dejar TODAS LAS PRUEBAS QUE HAGÁIS. </li>
        <li>Haz una prueba de predicción futura. Predice el dato en España en el último trimestre de 2027. Valora si puede estar prediciendo bien con los datos que tienes.</li>
    </ol>
</div>

In [1]:
from os import listdir
from numpy import asarray
from numpy import save
import tensorflow as tf
from tensorflow.keras.utils import load_img
from tensorflow.keras.utils import img_to_array
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from tensorflow import keras
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVR
from sklearn.metrics import classification_report
import datetime
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPool2D
from tensorflow.keras.layers import Flatten
from sklearn.decomposition import PCA

In [2]:
df = pd.read_csv('destruccion_empleo.csv')

In [3]:
df.head()

,record_id,country,iso3_code,region,income_group,year,quarter,quarter_label,industry_sector,sector_automation_risk_score,gdp_per_capita_usd,pct_sector_workforce_displaced,data_source_notes
0,1,United States,USA,North America,High Income,2020,1,2020-Q1,Technology & Software,0.382,63514,0.0406,Research-calibrated synthetic data. Grounded i...
1,2,United States,USA,North America,High Income,2020,1,2020-Q1,Finance & Banking,0.608,63514,0.0517,Research-calibrated synthetic data. Grounded i...
2,3,United States,USA,North America,High Income,2020,1,2020-Q1,Healthcare & Life Sciences,0.198,63514,0.0176,Research-calibrated synthetic data. Grounded i...
3,4,United States,USA,North America,High Income,2020,1,2020-Q1,Manufacturing & Industry,0.720,63514,0.0924,Research-calibrated synthetic data. Grounded i...
4,5,United States,USA,North America,High Income,2020,1,2020-Q1,Retail & E-Commerce,0.676,63514,0.0667,Research-calibrated synthetic data. Grounded i...


In [4]:

df.drop(['record_id','country','data_source_notes','quarter_label' ],axis=1,inplace=True)
#1.2 Elegir las entradas.
#1. Borrar estas 4. record_id es un identificador, country es redundante por iso3_code y quarter_label es redundante con quarter y year.

In [5]:
df['income_group'].unique()
#Encaja el label encoding. Se evita el 0 porque a la hora de operar tiene un efecto distinto
df_solucion = df.replace({'High Income':4,'Upper Middle Income':3,'Lower Middle Income':2, 'Low Income':1})
#1.1 En este caso, income_group, puede ser label o puede ser one hot. El resto pueden ir con get_dummies.

C:\Users\Equipo\AppData\Local\Temp\ipykernel_19452\781610323.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_solucion = df.replace({'High Income':4,'Upper Middle Income':3,'Lower Middle Income':2, 'Low Income':1})


In [6]:
#Ya tenemos todo limpio, se puede hacer get_dummies
df_solucion = pd.get_dummies(df_solucion,dtype=int)

In [7]:
df_solucion.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20800 entries, 0 to 20799
Columns: 108 entries, income_group to industry_sector_Transportation & Logistics
dtypes: float64(2), int64(106)
memory usage: 17.1 MB


In [8]:
y = df_solucion['pct_sector_workforce_displaced'].to_frame()
X = df_solucion.drop(['pct_sector_workforce_displaced'],axis=1)

In [9]:
df_solucion.corr()['pct_sector_workforce_displaced'].abs().sort_values(ascending=False)

pct_sector_workforce_displaced    1.000000
gdp_per_capita_usd                0.641091
income_group                      0.548548
sector_automation_risk_score      0.477408
year                              0.450259
                                    ...   
iso3_code_HUN                     0.008761
quarter                           0.008421
iso3_code_POL                     0.007610
iso3_code_CHN                     0.004474
iso3_code_GRC                     0.003983
Name: pct_sector_workforce_displaced, Length: 108, dtype: float64

In [10]:
#Viendo la correlación anterior, debemos predecir que tenemos un problema. Seguramente no 
# llegaremos a tener un buen resultado. Vamos a escalar los datos. Escalo tanto X como y porque, de lo contario, 
# surge el problema de que no entrena.
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X = scaler.fit_transform(X)
#y = scaler.fit_transform(y)


In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.1, random_state = 0)

In [12]:
#Vamos a probar con un Decision Tree a ver que resultado da para marcar un límite inferior de lo que debemos conseguir:
from sklearn.tree import DecisionTreeRegressor
tree_reg = DecisionTreeRegressor(random_state=42)
tree_reg.fit(X_train, y_train)
y_pred = tree_reg.predict(X_test)
from sklearn.metrics import r2_score
r2_score(y_test,y_pred)
#1.3 Diseñado la red, el entrenamiento y comprobado buen resultado 


0.9184200454893121

In [13]:
X.shape

(20800, 107)

In [ ]:
#1.4 Ha usado Técnicas de deep learning: batchnormalization, kernel_initializer y optimizers (al menos 3, si son 4 mejor)
#1.3 Diseñado la red, el entrenamiento y comprobado buen resultado 
model = keras.models.Sequential()
model.add(keras.layers.InputLayer(shape=X_train.shape[1:]))
model.add(keras.layers.Dense(64))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(16,activation='relu',kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(1,kernel_initializer='glorot_normal'))


model.compile(loss='mean_squared_error', optimizer = keras.optimizers.Adamax(learning_rate=0.00001, beta_1=0.9, beta_2=0.999), metrics=['mse'])

early_stopping_cb = keras.callbacks.EarlyStopping(patience=10,
restore_best_weights=True)
history = model.fit(X_train, y_train, epochs=100000,validation_split = 0.1,callbacks=[early_stopping_cb])

model.evaluate(X_test,y_test)

Epoch 1/100000
527/527 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - loss: 1.2857 - mse: 1.2857 - val_loss: 1.2304 - val_mse: 1.2304
Epoch 2/100000
527/527 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 1.1293 - mse: 1.1293 - val_loss: 1.0680 - val_mse: 1.0680
Epoch 3/100000
527/527 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.9788 - mse: 0.9788 - val_loss: 0.9253 - val_mse: 0.9253
Epoch 4/100000
527/527 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.8605 - mse: 0.8605 - val_loss: 0.7901 - val_mse: 0.7901
Epoch 5/100000
527/527 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - loss: 0.7526 - mse: 0.7526 - val_loss: 0.6922 - val_mse: 0.6922
Epoch 6/100000
527/527 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 0.6623 - mse: 0.6623 - val_loss: 0.6023 - val_mse: 0.6023
Epoch 7/100000
527/527 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 0.5841 - mse: 0.5841 - val_loss: 0.5216 - val_mse: 0.5216
Epoch 8/100000
527/527 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.5134 - mse: 0.5134 - val_loss: 0.4556 - val_mse: 0.4556
Epoch 9/100000
527/527 ━

[0.0003683316463138908, 0.0003683316463138908]

In [ ]:
model = keras.models.Sequential()
model.add(keras.layers.InputLayer(shape=X_train.shape[1:]))
model.add(keras.layers.Dense(64))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(16,activation='relu',kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(1,kernel_initializer='glorot_normal'))


model.compile(loss='mean_squared_error', optimizer = keras.optimizers.RMSprop(learning_rate=0.001, rho=0.9), metrics=['mse'])

early_stopping_cb = keras.callbacks.EarlyStopping(patience=10,restore_best_weights=True)
history = model.fit(X_train, y_train, epochs=100000,validation_split = 0.1,callbacks=[early_stopping_cb])

model.evaluate(X_test,y_test)

Epoch 1/100000
527/527 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.0409 - mse: 0.0409 - val_loss: 9.7178e-04 - val_mse: 9.7178e-04
Epoch 2/100000
527/527 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0011 - mse: 0.0011 - val_loss: 5.0697e-04 - val_mse: 5.0697e-04
Epoch 3/100000
527/527 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 6.2752e-04 - mse: 6.2752e-04 - val_loss: 2.0579e-04 - val_mse: 2.0579e-04
Epoch 4/100000
527/527 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 4.4257e-04 - mse: 4.4257e-04 - val_loss: 2.0486e-04 - val_mse: 2.0486e-04
Epoch 5/100000
527/527 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 3.8833e-04 - mse: 3.8833e-04 - val_loss: 2.8484e-04 - val_mse: 2.8484e-04
Epoch 6/100000
527/527 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 3.5269e-04 - mse: 3.5269e-04 - val_loss: 2.0177e-04 - val_mse: 2.0177e-04
Epoch 7/100000
527/527 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 3.1871e-04 - mse: 3.1871e-04 - val_loss: 2.5470e-04 - val_mse: 2.5470e-04
Epoch 8/100000
527/527 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms

[8.308862015837803e-05, 8.308862015837803e-05]

In [15]:
y_pred = model.predict(X_test)
r2_score(y_test,y_pred)

65/65 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 


0.8047083616256714

In [13]:
model = keras.models.Sequential()
model.add(keras.layers.InputLayer(shape=X_train.shape[1:]))
model.add(keras.layers.Dense(64))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(16,activation='relu',kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(1,kernel_initializer='glorot_normal'))


model.compile(loss='mean_squared_error', optimizer = keras.optimizers.Nadam(learning_rate=0.001, beta_1=0.9, beta_2=0.999), metrics=['mse'])

early_stopping_cb = keras.callbacks.EarlyStopping(patience=10,restore_best_weights=True)
history = model.fit(X_train, y_train, epochs=100000,validation_split = 0.1,callbacks=[early_stopping_cb])

model.evaluate(X_test,y_test)


Epoch 1/100000
527/527 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.0911 - mse: 0.0911 - val_loss: 0.0051 - val_mse: 0.0051
Epoch 2/100000
527/527 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0086 - mse: 0.0086 - val_loss: 0.0018 - val_mse: 0.0018
Epoch 3/100000
527/527 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0040 - mse: 0.0040 - val_loss: 7.1842e-04 - val_mse: 7.1842e-04
Epoch 4/100000
527/527 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0024 - mse: 0.0024 - val_loss: 6.7880e-04 - val_mse: 6.7880e-04
Epoch 5/100000
527/527 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0016 - mse: 0.0016 - val_loss: 4.7838e-04 - val_mse: 4.7838e-04
Epoch 6/100000
527/527 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0012 - mse: 0.0012 - val_loss: 4.5336e-04 - val_mse: 4.5336e-04
Epoch 7/100000
527/527 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 9.3117e-04 - mse: 9.3117e-04 - val_loss: 3.6844e-04 - val_mse: 3.6844e-04
Epoch 8/100000
527/527 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 7.4373e-04 - mse: 7.4373e-04 - val

[8.931724732974544e-05, 8.931724732974544e-05]

In [14]:
y_pred = model.predict(X_test)
r2_score(y_test,y_pred)

65/65 ━━━━━━━━━━━━━━━━━━━━ 0s 916us/step


0.9526434540748596

In [16]:
model = keras.models.Sequential()
model.add(keras.layers.InputLayer(shape=X_train.shape[1:]))
model.add(keras.layers.Dense(64))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(16,activation='relu',kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(1,kernel_initializer='glorot_normal'))


model.compile(loss='mean_squared_error', optimizer = keras.optimizers.Adam(learning_rate=0.01, beta_1=0.9, beta_2=0.999), metrics=['mse'])

early_stopping_cb = keras.callbacks.EarlyStopping(patience=10,
restore_best_weights=True)
history = model.fit(X_train, y_train, epochs=100000,validation_split = 0.1,callbacks=[early_stopping_cb])

model.evaluate(X_test,y_test)

Epoch 1/100000
527/527 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.0198 - mse: 0.0198 - val_loss: 4.3172e-04 - val_mse: 4.3172e-04
Epoch 2/100000
527/527 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 7.0067e-04 - mse: 7.0067e-04 - val_loss: 3.2583e-04 - val_mse: 3.2583e-04
Epoch 3/100000
527/527 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 5.4120e-04 - mse: 5.4120e-04 - val_loss: 3.8631e-04 - val_mse: 3.8631e-04
Epoch 4/100000
527/527 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - loss: 5.0322e-04 - mse: 5.0322e-04 - val_loss: 3.5731e-04 - val_mse: 3.5731e-04
Epoch 5/100000
527/527 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 4.2911e-04 - mse: 4.2911e-04 - val_loss: 3.2098e-04 - val_mse: 3.2098e-04
Epoch 6/100000
527/527 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 3.7725e-04 - mse: 3.7725e-04 - val_loss: 2.2057e-04 - val_mse: 2.2057e-04
Epoch 7/100000
527/527 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 3.1774e-04 - mse: 3.1774e-04 - val_loss: 1.5451e-04 - val_mse: 1.5451e-04
Epoch 8/100000
527/527 ━━━━━━━━━━━━━━━━━━━

[9.904980834107846e-05, 9.904980834107846e-05]

In [17]:
y_pred = model.predict(X_test)
r2_score(y_test,y_pred)

65/65 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


0.9474831819534302

In [ ]:
#1.5 ha realizado una predicción para el último trimestre de 2027
df_fila_nueva = df.loc[(df['iso3_code']=='ESP') & (df['year'] == 2026) & (df['industry_sector'] == 'Technology & Software') & (df['quarter'] == 2)]
df_fila_nueva['quarter'] = [4]
df_fila_nueva['year'] = [2027]

C:\Users\Equipo\AppData\Local\Temp\ipykernel_21096\4039567122.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_fila_nueva['quarter'] = [4]
C:\Users\Equipo\AppData\Local\Temp\ipykernel_21096\4039567122.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_fila_nueva['year'] = [2027]


In [19]:
df_fila_nueva

,iso3_code,region,income_group,year,quarter,industry_sector,sector_automation_risk_score,gdp_per_capita_usd,pct_sector_workforce_displaced
3370,ESP,Europe,High Income,2027,4,Technology & Software,0.366,30362,0.0829


In [20]:
df_con_fila_nueva = pd.concat([df,df_fila_nueva])

In [21]:
df_solucion_con_fila_nueva = df_con_fila_nueva.replace({'High Income':4,'Upper Middle Income':3,'Lower Middle Income':2, 'Low Income':1})

C:\Users\Equipo\AppData\Local\Temp\ipykernel_21096\99603624.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_solucion_con_fila_nueva = df_con_fila_nueva.replace({'High Income':4,'Upper Middle Income':3,'Lower Middle Income':2, 'Low Income':1})


In [22]:
df_solucion_con_fila_nueva = pd.get_dummies(df_solucion_con_fila_nueva,dtype=int)

In [23]:
X_fila = df_solucion_con_fila_nueva.drop(['pct_sector_workforce_displaced'],axis=1)

In [24]:
X_fila = scaler.transform(X_fila)

In [29]:
X_fila.shape

(20801, 107)

In [34]:
model.predict(X_fila[-1:])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step


array([[0.11455891]], dtype=float32)

<div style="border-style:groove;border-width:thin;padding:10px">
    <h4>EJERCICIO 2:</h4>
    <p>Descarga y descomprime el archivo 'DeFungi.zip'. Contiene imágenes microscópicas de hongos. Trata de conseguir los siguientes objetivos:</p>
    <ol>
        <li>Carga los datos de las imágenes preparándolos para que sean válidos para entrenar una IA.</li>
        <li>Resuelve el problema con una red neuronal convolucional. Puedes usar todas las técnicas vistas en clase y seguir un proceso correcto para solucionar el problema. Debéis dejar TODAS LAS PRUEBAS QUE HAGÁIS. Debes hacer, al menos, 2 pruebas y explicar que pasa cuando lo haces: coger menos imágenes y reducir la calidad de las imágenes cogiendo todas.</li>
    </ol>
</div>

In [15]:
#2.1 Cargar los datos según los dos métodos propuestos:
#1 A calidad completa pero pocas fotos. 50 de cada. Importante cogerlas por tipos.
photos =  []
labels = []
len_H1 = 0
len_H2 = 0
len_H3 = 0
len_H5 = 0
len_H6 = 0
for file in listdir('DeFungi/'):
    try:    
        if (file[:2] == 'H1') and (len_H1<50):
            photo = load_img('DeFungi/' + file, target_size=(500, 500))
            photos.append(img_to_array(photo))
            id = 0
            len_H1 += 1
            labels.append(float(id))
            del photo
        elif (file[:2] == 'H2') and (len_H2<50):
            photo = load_img('DeFungi/' + file, target_size=(500, 500))
            photos.append(img_to_array(photo))
            id = 1
            len_H2 += 1
            labels.append(float(id))
            del photo
        elif (file[:2] == 'H3') and (len_H3<50):
            photo = load_img('DeFungi/' + file, target_size=(500, 500))
            photos.append(img_to_array(photo))
            id = 2
            len_H3 += 1
            labels.append(float(id))
            del photo
        elif (file[:2] == 'H5') and (len_H5<50):
            photo = load_img('DeFungi/' + file, target_size=(500, 500))
            photos.append(img_to_array(photo))
            id = 3
            len_H5 += 1
            labels.append(float(id))
            del photo
        elif (file[:2] == 'H6') and (len_H6<50):
            photo = load_img('DeFungi/' + file, target_size=(500, 500))
            photos.append(img_to_array(photo))
            id = 4
            len_H6 += 1
            labels.append(float(id))
            del photo
        
    except:
        print('Error en la foto ' + file)

In [16]:
X = asarray(photos)/255.
y = asarray(labels)

In [17]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.1, random_state = 0)

In [18]:
from sklearn.ensemble import RandomForestClassifier
rnd_clf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=0)
rnd_clf.fit(X_train.reshape(X_train.shape[0],-1),y_train)
from sklearn.metrics import accuracy_score
y_pred = rnd_clf.predict(X_test.reshape(X_test.shape[0],-1))
print(accuracy_score(y_test,y_pred))

0.52


In [ ]:

#2.2 Diseño de la red convolucional y del entrenamiento.
model = keras.models.Sequential()
model.add(Conv2D(16,(3,3), activation='relu', input_shape = (500,500,3)))
model.add(MaxPool2D(2,2))
model.add(Conv2D(32,(3,3), activation='relu'))
model.add(MaxPool2D(2,2))
model.add(Conv2D(64,(3,3), activation='relu'))
model.add(MaxPool2D(2,2))
model.add(Conv2D(256,(3,3), activation='relu'))
model.add(MaxPool2D(2,2))
model.add(Flatten())
#Haciendo summary se ve que de la flatten salen 215000 salidas. Como es imposible poner 140000 neuronas, pongo muchas pero menos.
model.add(keras.layers.Dense(2048,activation='relu',kernel_initializer='he_normal'))
model.add(keras.layers.Dense(512,activation='relu',kernel_initializer='he_normal'))
model.add(keras.layers.Dense(5,activation='softmax',kernel_initializer='glorot_normal'))

c:\Users\Equipo\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.compile(loss='sparse_categorical_crossentropy', optimizer = keras.optimizers.Adam(learning_rate=0.0001, beta_1=0.9, beta_2=0.999), metrics=['accuracy'])
early_stopping_cb = keras.callbacks.EarlyStopping(patience=5,
restore_best_weights=True)
history = model.fit(X_train, y_train, epochs=2,validation_split = 0.1,callbacks=[early_stopping_cb],batch_size=256)

Epoch 1/2
1/1 ━━━━━━━━━━━━━━━━━━━━ 15s 15s/step - accuracy: 0.2327 - loss: 1.6080 - val_accuracy: 0.2174 - val_loss: 1.6087
Epoch 2/2
1/1 ━━━━━━━━━━━━━━━━━━━━ 12s 12s/step - accuracy: 0.1980 - loss: 1.5998 - val_accuracy: 0.2174 - val_loss: 1.6470


In [10]:
model.evaluate(X_test,y_test)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 314ms/step - accuracy: 0.2000 - loss: 1.6343


[1.6342763900756836, 0.20000000298023224]

In [ ]:
#2.1 bajando calidad. En este caso no hace falta bajar muchísimo.
photos =  []
labels = []
len_H1 = 0
len_H2 = 0
len_H3 = 0
len_H5 = 0
len_H6 = 0
for file in listdir('DeFungi/'):
    try:    
        if (file[:2] == 'H1') and (len_H1<300):
            photo = load_img('DeFungi/' + file, target_size=(100, 100))
            photos.append(img_to_array(photo))
            id = 0
            len_H1 += 1
            labels.append(float(id))
            del photo
        elif (file[:2] == 'H2') and (len_H2<300):
            photo = load_img('DeFungi/' + file, target_size=(100, 100))
            photos.append(img_to_array(photo))
            id = 1
            len_H2 += 1
            labels.append(float(id))
            del photo
        elif (file[:2] == 'H3') and (len_H3<300):
            photo = load_img('DeFungi/' + file, target_size=(100, 100))
            photos.append(img_to_array(photo))
            id = 2
            len_H3 += 1
            labels.append(float(id))
            del photo
        elif (file[:2] == 'H5') and (len_H5<300):
            photo = load_img('DeFungi/' + file, target_size=(100, 100))
            photos.append(img_to_array(photo))
            id = 3
            len_H5 += 1
            labels.append(float(id))
            del photo
        elif (file[:2] == 'H6') and (len_H6<300):
            photo = load_img('DeFungi/' + file, target_size=(100, 100))
            photos.append(img_to_array(photo))
            id = 4
            len_H6 += 1
            labels.append(float(id))
            del photo
        
    except:
        print('Error en la foto ' + file)

In [33]:
X = asarray(photos)/255.
y = asarray(labels)

In [34]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.1, random_state = 0)

In [35]:
from sklearn.ensemble import RandomForestClassifier
rnd_clf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=0)
rnd_clf.fit(X_train.reshape(X_train.shape[0],-1),y_train)
from sklearn.metrics import accuracy_score
y_pred = rnd_clf.predict(X_test.reshape(X_test.shape[0],-1))
print(accuracy_score(y_test,y_pred))

0.5666666666666667


In [36]:

model = keras.models.Sequential()
model.add(Conv2D(16,(3,3), activation='relu', input_shape = (100,100,3)))
model.add(MaxPool2D(2,2))
model.add(Conv2D(32,(3,3), activation='relu'))
model.add(MaxPool2D(2,2))
model.add(Conv2D(64,(3,3), activation='relu'))
model.add(MaxPool2D(2,2))
model.add(Conv2D(256,(3,3), activation='relu'))
model.add(MaxPool2D(2,2))
model.add(Flatten())
model.add(keras.layers.Dense(256,activation='relu',kernel_initializer='he_normal'))
model.add(keras.layers.Dense(64,activation='relu',kernel_initializer='he_normal'))
model.add(keras.layers.Dense(5,activation='softmax',kernel_initializer='glorot_normal'))

c:\Users\Equipo\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [21]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 498, 498, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 249, 249, 16)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 247, 247, 32)   │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 123, 123, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 121, 121, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 60, 60, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 58, 58, 256)    │       147,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 29, 29, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 215296)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 100)            │    21,529,700 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 20)             │         2,020 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 5)              │           105 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,703,121 (82.79 MB)

 Trainable params: 21,703,121 (82.79 MB)

 Non-trainable params: 0 (0.00 B)

In [37]:
#Usamos Nadam. Se podrían usar otros optimizadores, pero el resultado es similar.
model.compile(loss='sparse_categorical_crossentropy', optimizer = keras.optimizers.Adam(learning_rate=0.0001, beta_1=0.9, beta_2=0.999), metrics=['accuracy'])
early_stopping_cb = keras.callbacks.EarlyStopping(patience=10,
restore_best_weights=True)
history = model.fit(X_train, y_train, epochs=1000,validation_split = 0.1,callbacks=[early_stopping_cb],batch_size=256)

Epoch 1/1000
5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 409ms/step - accuracy: 0.2395 - loss: 1.6071 - val_accuracy: 0.2815 - val_loss: 1.5989
Epoch 2/1000
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 372ms/step - accuracy: 0.3391 - loss: 1.5937 - val_accuracy: 0.3926 - val_loss: 1.5861
Epoch 3/1000
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 363ms/step - accuracy: 0.3934 - loss: 1.5773 - val_accuracy: 0.3556 - val_loss: 1.5666
Epoch 4/1000
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 357ms/step - accuracy: 0.3753 - loss: 1.5547 - val_accuracy: 0.3630 - val_loss: 1.5418
Epoch 5/1000
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 326ms/step - accuracy: 0.3877 - loss: 1.5247 - val_accuracy: 0.3852 - val_loss: 1.5081
Epoch 6/1000
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 323ms/step - accuracy: 0.3967 - loss: 1.4828 - val_accuracy: 0.3926 - val_loss: 1.4675
Epoch 7/1000
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 321ms/step - accuracy: 0.4107 - loss: 1.4323 - val_accuracy: 0.4963 - val_loss: 1.4294
Epoch 8/1000
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 356ms/step - accuracy: 0.4979 - loss: 1.3827 - val_accuracy: 0.

In [38]:
model.evaluate(X_test,y_test)

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.7133 - loss: 0.6896


[0.6896077394485474, 0.7133333086967468]